# Lab 6 - Task 1: Convolution & Histogram Equalization

**Objectives:**
1. Apply histogram equalization to an image and observe the effect on contrast.
2. Compare the histograms before and after equalization.
3. Discuss why histogram equalization improves contrast in images with poor lighting.

In [ ]:
import cv2
import numpy as np
import copy
import matplotlib.pyplot as plt

## Helper Functions

In [ ]:
def box_filt(n):
    """
    Creates a box filter kernel of size n x n.
    """
    if n <= 0 or n % 2 == 0:
        raise ValueError("Window size must be a positive odd integer.")
    kernel = np.ones((n, n), np.float32) / (n * n)
    return kernel


def apply_filters(image_input, box, filt_size):
    pad_size = int(np.ceil(filt_size / 2))
    image_padded = np.pad(
        image_input,
        pad_width=((pad_size, pad_size), (pad_size, pad_size)),
        mode='symmetric'
    )
    image_box = copy.deepcopy(image_input)
    row, column = image_input.shape
    for i in range(row):
        for j in range(column):
            patch_curr = image_padded[i:i + filt_size, j:j + filt_size]
            results_box = box * patch_curr
            image_box[i, j] = np.sum(results_box)
    return image_box

## Part 1A: Convolution with Sobel Kernel

In [ ]:
# Load image in grayscale
image = cv2.imread('image.png', cv2.IMREAD_GRAYSCALE)

if image is None:
    # Create a synthetic test image if no image file is found
    image = np.random.randint(50, 200, (256, 256), dtype=np.uint8)
    print("No image file found — using a synthetic test image.")

# Sobel edge-detection kernel
kernel = np.array([[1, 2, 1],
                   [0, 0, 0],
                   [-1, -2, -1]], dtype=np.float32)

convolved_image = cv2.filter2D(image, -1, kernel)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(image, cmap='gray')
axes[0].set_title('Original Image')
axes[0].axis('off')
axes[1].imshow(convolved_image, cmap='gray')
axes[1].set_title('Convolved (Sobel Kernel)')
axes[1].axis('off')
plt.tight_layout()
plt.show()

## Part 1B: Box Filter via Custom Convolution

In [ ]:
filt_size = 5
box = box_filt(filt_size)

image_float = image.astype(np.float32) / 255.0
image_box_filtered = apply_filters(image_float, box, filt_size)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(image_float, cmap='gray')
axes[0].set_title('Original (normalized)')
axes[0].axis('off')
axes[1].imshow(image_box_filtered, cmap='gray')
axes[1].set_title(f'Box Filter {filt_size}x{filt_size}')
axes[1].axis('off')
plt.tight_layout()
plt.show()

## Part 1C: Histogram Equalization

In [ ]:
# Apply histogram equalization
equalized_image = cv2.equalizeHist(image)

# Display original vs equalized
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(image, cmap='gray')
axes[0].set_title('Original Image')
axes[0].axis('off')
axes[1].imshow(equalized_image, cmap='gray')
axes[1].set_title('Histogram Equalized')
axes[1].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Compare histograms before and after equalization
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(image.ravel(), bins=256, range=(0, 256), color='steelblue')
axes[0].set_title('Histogram - Original')
axes[0].set_xlabel('Pixel Intensity')
axes[0].set_ylabel('Frequency')

axes[1].hist(equalized_image.ravel(), bins=256, range=(0, 256), color='darkorange')
axes[1].set_title('Histogram - Equalized')
axes[1].set_xlabel('Pixel Intensity')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

## Discussion

**Why does histogram equalization improve contrast in poorly-lit images?**

In a poorly-lit image, most pixel intensities are clustered in a narrow range (e.g., all dark or all bright). The histogram is therefore concentrated in one region and most intensity levels are unused.

Histogram equalization redistributes the pixel intensities so they span the full range [0, 255] more uniformly. It achieves this by computing the **cumulative distribution function (CDF)** of the original histogram and using it as a mapping function:

$$s_k = (L-1) \cdot CDF(r_k)$$

where $r_k$ are the original intensity levels and $s_k$ are the equalized levels.

The result is that subtle differences in intensity become more visible, revealing detail that was hidden in the compressed tonal range — effectively enhancing the global contrast of the image.